# Regime v2 Sanity Review

Use this notebook to review a generated `regime-multi-v1` artifact before promoting calibration changes. It is intentionally review-oriented: load snapshots, inspect the latest ensemble call, check known historical windows, and generate the standalone HTML report for side-by-side feedback.

**校准问题:** 这次 review 的目标是否只聚焦 macro_regime 和 market_regime，而不再接受 legacy 解释？

## What Regime v2 Is Classifying

Regime v2 reduces the macro state to two signed axes: **growth** and **inflation**. The quadrant mapping is deterministic:

| Growth sign | Inflation sign | Quadrant | Intuition |
|---|---|---|---|
| positive | negative | Goldilocks | growth support with easing/disinflation pressure |
| positive | positive | Reflation | growth support with rising inflation/rates pressure |
| negative | positive | Stagflation | weak growth with inflation pressure |
| negative | negative | Deflationary Slowdown | weak growth with disinflation/deflation pressure |

A **crisis overlay** is separate from the quadrant. A crisis day can still be tagged as Stagflation or Deflationary Slowdown; the overlay is the risk-off dial used by policy, not a replacement regime.

**校准问题:** 四象限定义是否符合你想用来指导 portfolio/risk policy 的经济直觉？

## Method 1: `macro_regime`

`macro_regime` is the macro fundamentals method. It consumes the FRED panel configured in `configs/regime_detection/fred_series.yml` or the example file. Each series is transformed, lagged by an approximate publication delay, forward-filled to business days, mapped into a signed raw signal, grouped into fast/slow buckets, and averaged into growth or inflation. Fast buckets carry 70% default weight and slow buckets carry 30%; optional z-score normalization is configurable but not the default.

**Growth axis inputs** currently include unemployment rate YoY change inverted, non-farm payrolls YoY, industrial production YoY, retail sales YoY, and the state coincident leading index diffusion series.

**Inflation axis inputs** currently include CPI YoY, core CPI YoY, PCE YoY, core PCE YoY, sticky CPI, 5y5y forward inflation expectations, and average hourly earnings YoY.

**Judgment rules:**

- positive growth score means the signed macro growth basket is positive; negative means growth evidence is weak.
- positive inflation score means inflation pressure evidence is positive; negative means disinflation evidence dominates.
- fast/slow bucket weights are configurable and default to 70/30 per axis.
- `min_consecutive_days` applies sign hysteresis so single-release flips do not immediately change the quadrant.
- this method does **not** emit crisis flags; risk-on/risk-off comes from `market_regime`.

**校准问题:** macro_regime 的 fast/slow 分组和 70/30 权重，是否需要按 growth 与 inflation 分别调整？

## Method 2: `market_regime`

`market_regime` reads market prices and volatility proxies from the cached Yahoo Finance market panel. It estimates growth from equity, sector leadership, commodity, credit, and defensive-relative-performance signals; inflation from oil/energy/commodity and inflation-sensitive proxies; and risk-on/risk-off from VIX, MOVE, credit stress, drawdown, and flight-to-quality proxies.

**Default interpretation:**

| Evidence | Axis / overlay impact |
|---|---|
| Broad equity and cyclicals outperforming defensives | growth positive |
| Oil, energy, broad commodities, or inflation-sensitive proxies rising | inflation positive |
| Treasury defensives outperforming equity or credit weakening | growth negative / risk-off evidence |
| VIX/MOVE/realized vol spikes | risk-off overlay |

**Scoring:** market signals use z-score/relative-return normalization by default because prices, vol levels, and spreads are not naturally comparable. Raw sign rules are available for explicit momentum diagnostics.

**校准问题:** market_regime 的 growth、inflation、risk-off 市场代理是否有缺失或权重明显不合理的地方？

## Ensemble and Policy Interpretation

The ensemble aligns enabled methods on common dates, votes separately on growth and inflation signs, weights by method weight and method confidence, carries exact ties forward, then applies another sign-hysteresis pass. `method_agreement` is the share of methods whose quadrant matches the final ensemble quadrant.

**Review interpretation:**

- `method_agreement = 1.0` with two methods means macro and market evidence agree on the quadrant.
- `method_agreement = 0.5` with two methods means disagreement; inspect method votes and native labels before drawing conclusions.
- missing one method means the ensemble may still run, but the output should be read as degraded coverage.
- crisis flag/intensity should line up with acute market-stress windows such as March 2020, not ordinary slowdowns.
- policy is resolved from the ensemble quadrant first; the risk-off overlay then reduces the vol multiplier and shifts a fraction of EQ into CASH/GOLD/FI.

**校准问题:** 当 macro 和 market 分歧时，你希望 ensemble 更偏慢变量确认，还是更偏市场快速反应？

## Human Review Checklist

Use these checks before accepting calibration changes:

1. Latest state: does the latest quadrant match the visible macro/market backdrop, and is method agreement high enough for the intended policy use?
2. Historical anchors: 2017 should generally look like Goldilocks/Reflation; March 2020 should activate crisis overlay; 2022 should show inflation pressure through Reflation/Stagflation-like states; late 2023 should show disinflation/recovery behavior if the macro panel supports it.
3. Transition quality: labels should not churn on single data releases; duration should show persistence except during genuine regime transitions.
4. Axis reasonableness: growth and inflation scores should not stay saturated for implausibly long periods; if they do, revisit z-score windows, clipping, and series transforms.
5. Data coverage: confirm both `macro_regime` and `market_regime` are present for periods where the ensemble is expected to be full-strength.
6. Policy sanity: the resolved vol multiplier and target buckets should be plausible for the quadrant and crisis intensity.

**校准问题:** 这份 checklist 里的历史锚点和验收标准是否足够，是否还要加入其他时期？

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import market_helper
import pandas as pd
from IPython.display import HTML, display

from market_helper.reporting.regime_html import write_regime_html_report
from market_helper.workflows.run_regime_report import refresh_data_and_run_regime_report, run_regime_report_from_existing_data

REPO_ROOT = Path(market_helper.__file__).resolve().parents[1]

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# 校准问题: 当前 import 依赖是否覆盖你希望 notebook 展示的所有校准视图？

## Paths

Adjust these paths to point at the artifacts you want to review. The notebook has the same two operating modes as the CLI: refresh stale source data then run, or reuse existing source data and only rerun regime detection/reporting.

**校准问题:** 这些默认路径是否匹配你本地实际要 review 的 artifacts？

In [ ]:
REGIME_JSON = REPO_ROOT / "data/artifacts/regime_detection/regime_snapshots.json"
HTML_OUTPUT = REPO_ROOT / "data/artifacts/regime_detection/regime_report.html"

MACRO_PANEL = REPO_ROOT / "data/interim/fred/macro_panel.feather"
FRED_SERIES_CONFIG = REPO_ROOT / "configs/regime_detection/fred_series.yml"
if not FRED_SERIES_CONFIG.exists():
    FRED_SERIES_CONFIG = REPO_ROOT / "configs/regime_detection/fred_series.example.yml"
MARKET_PANEL = REPO_ROOT / "data/interim/market_regime/market_panel.feather"
MARKET_REGIME_CONFIG = REPO_ROOT / "configs/regime_detection/market_regime.yml"
if not MARKET_REGIME_CONFIG.exists():
    MARKET_REGIME_CONFIG = REPO_ROOT / "configs/regime_detection/market_regime.example.yml"

REFRESH_SOURCE_DATA = False
RUN_DETECTION = not REGIME_JSON.exists()
MAX_AGE_DAYS = 7
METHODS = ("macro_regime", "market_regime")

# 校准问题: 本次 review 要不要重新跑 detection，还是只检查已有 artifact？

## Optional: refresh/run entry points

Set `RUN_DETECTION = True` to rerun regime detection and regenerate HTML. If `REFRESH_SOURCE_DATA = True`, the notebook first refreshes stale FRED macro and Yahoo market-panel artifacts using `MAX_AGE_DAYS` as the freshness window. If `REFRESH_SOURCE_DATA = False`, it uses existing local source artifacts only.

**校准问题:** 你希望这次 review 使用 refresh 模式拉新数据，还是固定当前本地数据做可复现校准？

In [ ]:
if RUN_DETECTION and REFRESH_SOURCE_DATA:
    result = refresh_data_and_run_regime_report(
        methods=METHODS,
        max_age_days=MAX_AGE_DAYS,
        macro_panel_path=MACRO_PANEL,
        fred_series_config=FRED_SERIES_CONFIG,
        market_panel_path=MARKET_PANEL,
        market_regime_config=MARKET_REGIME_CONFIG,
        output_regime_path=REGIME_JSON,
        output_html_path=HTML_OUTPUT,
    )
    print(result)
elif RUN_DETECTION:
    result = run_regime_report_from_existing_data(
        methods=METHODS,
        macro_panel_path=MACRO_PANEL,
        fred_series_config=FRED_SERIES_CONFIG,
        market_panel_path=MARKET_PANEL,
        market_regime_config=MARKET_REGIME_CONFIG,
        output_regime_path=REGIME_JSON,
        output_html_path=HTML_OUTPUT,
    )
    print(result)
else:
    print("RUN_DETECTION is False; using existing artifact if present.")

# 校准问题: 运行结果是否成功生成 macro_panel、market_panel、regime JSON 和 HTML；如果失败，缺的是哪个输入？

In [ ]:
if not REGIME_JSON.exists():
    raise FileNotFoundError(
        f"Missing regime artifact: {REGIME_JSON}. Run the previous cell with RUN_DETECTION=True; "
        "set REFRESH_SOURCE_DATA=True if the macro or market source panels are also missing."
    )

payload = json.loads(REGIME_JSON.read_text(encoding="utf-8"))
if not isinstance(payload, list) or not payload:
    raise ValueError("Regime artifact must be a non-empty JSON array")

latest = payload[-1]
schema = latest.get("version", "regime-v1") if isinstance(latest, dict) else "unknown"
print(f"rows={len(payload):,} schema={schema} latest_as_of={latest.get('as_of')}")

# 校准问题: 当前 artifact 的行数、schema、latest_as_of 是否符合你对数据新鲜度的要求？

In [ ]:
def snapshots_to_frame(rows: list[dict]) -> pd.DataFrame:
    out = []
    for row in rows:
        if not isinstance(row, dict):
            continue
        if row.get("version") == "regime-multi-v1" or isinstance(row.get("ensemble"), dict):
            ensemble = row.get("ensemble") or {}
            axes = ensemble.get("axes") or {}
            diagnostics = ensemble.get("diagnostics") or {}
            out.append(
                {
                    "as_of": pd.to_datetime(row.get("as_of")),
                    "regime": ensemble.get("quadrant"),
                    "growth_score": axes.get("growth_score"),
                    "inflation_score": axes.get("inflation_score"),
                    "method_agreement": diagnostics.get("method_agreement"),
                    "crisis_flag": ensemble.get("crisis_flag"),
                    "crisis_intensity": ensemble.get("crisis_intensity"),
                    "duration_days": ensemble.get("duration_days"),
                    "methods": ", ".join(sorted((row.get("per_method") or {}).keys())),
                }
            )
        else:
            raise ValueError("Notebook expects regime-multi-v1 snapshots from macro_regime + market_regime")
    return pd.DataFrame(out).sort_values("as_of").reset_index(drop=True)

frame = snapshots_to_frame(payload)
frame.tail(10)

# 校准问题: frame 里的 growth/inflation score、method agreement、risk overlay 字段是否足够支持校准判断？

## Method-level Macro vs Market Views

The ensemble can hide method disagreement. This section expands `per_method` so `macro_regime` and `market_regime` are visible side by side.

**校准问题:** `market_regime` 的 quadrant、growth score、inflation score、risk overlay 是否和 ensemble 的解释一致？

In [ ]:
def method_snapshots_to_frame(rows: list[dict]) -> pd.DataFrame:
    out = []
    for row in rows:
        if not isinstance(row, dict):
            continue
        per_method = row.get("per_method") or {}
        for method_name, method_payload in sorted(per_method.items()):
            quadrant = method_payload.get("quadrant") or {}
            axes = quadrant.get("axes") or {}
            diagnostics = quadrant.get("diagnostics") or {}
            out.append(
                {
                    "as_of": pd.to_datetime(row.get("as_of")),
                    "method": method_name,
                    "quadrant": quadrant.get("quadrant"),
                    "growth_score": axes.get("growth_score"),
                    "inflation_score": axes.get("inflation_score"),
                    "confidence": axes.get("confidence"),
                    "crisis_flag": quadrant.get("crisis_flag"),
                    "crisis_intensity": quadrant.get("crisis_intensity"),
                    "risk_regime": diagnostics.get("risk_regime"),
                    "risk_score": diagnostics.get("risk_score"),
                    "duration_days": quadrant.get("duration_days"),
                    "native_label": method_payload.get("native_label"),
                }
            )
    return pd.DataFrame(out).sort_values(["as_of", "method"]).reset_index(drop=True)

method_frame = method_snapshots_to_frame(payload)
latest_method_frame = method_frame[method_frame["as_of"] == method_frame["as_of"].max()].copy()
display(latest_method_frame.T)
display(method_frame.groupby("method")[["growth_score", "inflation_score", "confidence", "crisis_intensity", "risk_score"]].mean(numeric_only=True).round(3))

# 校准问题: 哪个 method 正在主导当前 ensemble；market_regime 是否被清楚展示出来？

## Market Regime Drivers

This cell expands `market_regime` drivers from the latest snapshot: growth drivers, inflation drivers, and risk overlay drivers.

**校准问题:** market growth、inflation、risk-off 的 top drivers 是否符合你看到的市场状态？

In [ ]:
def latest_method_payload(method_name: str) -> dict:
    per_method = latest.get("per_method") or {}
    if method_name not in per_method:
        raise KeyError(f"{method_name} not found in latest snapshot. Available: {sorted(per_method)}")
    return per_method[method_name]


def driver_frame(method_payload: dict, *, method_name: str) -> pd.DataFrame:
    quadrant = method_payload.get("quadrant") or {}
    axes = quadrant.get("axes") or {}
    diagnostics = quadrant.get("diagnostics") or {}
    rows = []
    for axis, drivers in (
        ("growth", axes.get("growth_drivers") or {}),
        ("inflation", axes.get("inflation_drivers") or {}),
        ("risk", diagnostics.get("risk_drivers") or {}),
    ):
        for name, value in sorted(drivers.items(), key=lambda item: abs(float(item[1])), reverse=True):
            rows.append({"method": method_name, "axis": axis, "driver": name, "contribution": float(value)})
    return pd.DataFrame(rows)

market_latest = latest_method_payload("market_regime")
market_driver_frame = driver_frame(market_latest, method_name="market_regime")
display(pd.DataFrame([market_latest.get("quadrant", {})]).T)
display(market_driver_frame)

# 校准问题: market_regime 的最大正负贡献项是否需要调权、移除或增加替代 ticker？

## Macro Regime Buckets And Drivers

This cell expands `macro_regime` bucket scores and driver contributions so macro fast/slow evidence is visible alongside market evidence.

**校准问题:** macro fast/slow bucket 是否按 70/30 反映了你对快慢数据可靠性的判断？

In [ ]:
macro_latest = latest_method_payload("macro_regime")
macro_quadrant = macro_latest.get("quadrant") or {}
macro_diagnostics = macro_quadrant.get("diagnostics") or {}
macro_driver_frame = driver_frame(macro_latest, method_name="macro_regime")

bucket_scores = macro_diagnostics.get("bucket_scores") or {}
bucket_rows = []
for axis, buckets in bucket_scores.items():
    for bucket, value in buckets.items():
        bucket_rows.append({"axis": axis, "bucket": bucket, "score": value})

display(pd.DataFrame(bucket_rows))
display(macro_driver_frame)

# 校准问题: macro_regime 是 fast bucket 还是 slow bucket 在驱动当前判断；这个方向是否合理？

## Latest call and distribution

**校准问题:** 最新 regime 和分布是否需要从业务角度进一步解释或拆分？

In [ ]:
display(frame.tail(1).T)
display(frame["regime"].value_counts(dropna=False).rename("count").to_frame())
display(frame.groupby("regime", dropna=False)[["growth_score", "inflation_score", "method_agreement", "crisis_intensity"]].mean(numeric_only=True).round(3))

# 校准问题: latest call 是否符合你对当前宏观和市场状态的判断？如果不符合，是哪个 axis 错了？

## Historical checkpoint review

These dates are not pass/fail tests. They are anchors for human review after calibration changes.

**校准问题:** 这些 checkpoint 日期是否覆盖了你最关心的 regime 转折？

In [ ]:
CHECKPOINTS = {
    "2017 Goldilocks/Reflation sanity": "2017-06-30",
    "COVID crisis overlay sanity": "2020-03-20",
    "2022 inflation regime sanity": "2022-06-30",
    "2023 disinflation/recovery sanity": "2023-11-30",
}

checkpoint_rows = []
for label, date_text in CHECKPOINTS.items():
    target = pd.Timestamp(date_text)
    idx = (frame["as_of"] - target).abs().idxmin()
    row = frame.loc[idx].to_dict()
    row["checkpoint"] = label
    row["target_date"] = target.date().isoformat()
    checkpoint_rows.append(row)

checkpoint_frame = pd.DataFrame(checkpoint_rows)[[
    "checkpoint",
    "target_date",
    "as_of",
    "regime",
    "growth_score",
    "inflation_score",
    "method_agreement",
    "crisis_flag",
    "crisis_intensity",
    "methods",
]]
checkpoint_frame

# 校准问题: 每个 checkpoint 的 quadrant、risk overlay、method agreement 是否符合预期；不符合时应调整哪个信号或权重？

## Generate standalone HTML

**校准问题:** HTML 报告是否是你希望后续作为 review artifact 的格式？

In [ ]:
html_path = write_regime_html_report(regime_path=REGIME_JSON, output_path=HTML_OUTPUT)
print(html_path)
display(HTML(f"<a href='{html_path.as_posix()}' target='_blank'>Open regime HTML report</a>"))

# 校准问题: 生成的 HTML 是否清楚展示了你需要回答的 calibration evidence？